# 03 · DICOM ön işleme: kesitleri doğru sırada, doğru yönde, doğru ölçekte okumak

**Bu notebook ne yapar?** Bir test vakasının ham DICOM dosyaları üzerinde `src/dicom.py` ve
`src/cache.py` modüllerinin yaptığı her adımı görsel olarak gösterir. Eğitim ve inference
aynı fonksiyonları kullanır; burada doğru çalıştıklarını gözle doğruluyoruz.

**Neden önemli?** Eski pipeline'da dört sessiz hata vardı. Hiçbiri hata mesajı üretmez;
model sadece kötü öğrenir:

| # | tuzak | eski kod | yeni kod |
|---|---|---|---|
| 1 | kesit sırası | dosya adına göre sıralama (dosya adı = rastgele UID) | `ImagePositionPatient` · kesit normali |
| 2 | görüntü yönü | yok | yön kosinüslerine göre çevirme |
| 3 | fiziksel ölçek | her görüntü sabit piksele sündürülüyordu | sabit **160 mm** görüş alanı → 224 px (0.71 mm/px) |
| 4 | sağ / sol diz | yok | sağ dizler "sol diz" görünümüne aynalanır |
| + | yoğunluk | her kesit ayrı normalize | **seri** düzeyinde tek pencere |

**Koordinat sistemi (DICOM / LPS):** +x hastanın **soluna**, +y **arkaya**, +z **başa** doğru.

> Yerelde çalıştırmak için `data/test_series/<study>/` altında DICOM gerekir
> (bkz. README: tek bir test vakası Kaggle API ile indirilebilir).

In [ ]:
# Proje kökünü (src/ klasörünü içeren yer) Python yoluna ekle: yerelde ve Kaggle'da çalışır.
import os, sys
from pathlib import Path

def add_project_root():
    cands = [Path.cwd(), Path.cwd().parent]
    if Path("/kaggle/input").exists():
        # os.walk + budama: yarışmanın DICOM klasörlerine girmez (rglob dakikalar sürüyordu)
        for d, dirs, files in os.walk("/kaggle/input"):
            dirs[:] = [x for x in dirs if x not in ("train_series", "test_series")]
            if Path(d).name == "src" and "__init__.py" in files:
                cands.append(Path(d).parent)
    for c in cands:
        if (c / "src" / "__init__.py").exists():
            sys.path.insert(0, str(c))
            return c
    raise FileNotFoundError("src/ bulunamadı")

PROJECT_ROOT = add_project_root()
print("Proje kökü:", PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom
from scipy.stats import spearmanr

from src.config import DATA_DIR
from src.dicom import analyze_series, load_series_volume, resolve_laterality, _read_pixels
from src.cache import CacheConfig, process_study, SLOT_NAMES

series_df = pd.read_csv(DATA_DIR / "test_series.csv")
local = [s for s in series_df.StudyInstanceUID.unique() if (DATA_DIR / "test_series" / s).exists()]
STUDY = local[0]
study_rows = series_df[series_df.StudyInstanceUID == STUDY].reset_index(drop=True)
print("Vaka:", STUDY)
study_rows

## 1. Kesit sırası: dosya adı anatomik sırayı vermez

Her serinin her dosyası için iki sıralama karşılaştırıyoruz:
* **dosya adı sırası** (eski kod: `sorted(glob("*.dcm"))`)
* **fiziksel konum**: `ImagePositionPatient`'ın, kesit normalinin baskın ekseni (sagital → x,
  koronal → y, aksiyal → z) üzerindeki değeri.

Spearman korelasyonu 1 olsaydı dosya adı sırası doğru olurdu.

In [ ]:
geoms, rows = {}, []
for _, r in study_rows.iterrows():
    d = DATA_DIR / "test_series" / STUDY / r.SeriesInstanceUID
    g = analyze_series(d)
    geoms[r.SeriesInstanceUID] = g
    by_name = sorted(g.headers, key=lambda h: h.path.name)
    pos_name = [h.ipp[g.normal_axis] for h in by_name]
    inst = [h.instance for h in by_name]
    ds = pydicom.dcmread(str(g.headers[0].path), stop_before_pixels=True)
    rows.append({
        "CSV düzlem": r.Anatomical_Plane, "geometri düzlem": g.plane, "FS": r.Fluid_Sensitive,
        "kesit": len(g.headers),
        "ρ(dosya adı, konum)": spearmanr(range(len(pos_name)), pos_name)[0],
        "ρ(InstanceNumber, konum)": spearmanr(inst, pos_name)[0],
        "piksel mm": round(float(g.headers[0].pixel_spacing[0]), 3),
        "matris": f"{g.headers[0].rows}x{g.headers[0].cols}",
        "FOV mm": round(float(g.headers[0].pixel_spacing[0] * g.headers[0].rows), 1),
        "açıklama": getattr(ds, "SeriesDescription", ""),
    })
pd.DataFrame(rows).round(2)

**Okuma:**
* `ρ(dosya adı, konum)` sıfıra yakın → eski kod kesitleri **karışık** sırayla okuyordu.
  "Ardışık üç kesit = 2.5D pencere" fikri, birbiriyle ilgisiz üç kesit demekti.
* `ρ(InstanceNumber, konum)` ±1: sıra doğru ama **yön** cihaza göre ters olabiliyor (−1).
  Bu yüzden yön bağımsız olan geometriyi kullanıyoruz.
* Piksel boyutu ve FOV seriden seriye değişiyor → sabit mm ölçeği gerekli (bölüm 3).

Aşağıda aynı sagital serinin ortasından "ardışık" üç kesit: solda dosya adı sırasıyla,
sağda geometrik sırayla.

In [ ]:
sag_id = next(k for k, g in geoms.items() if g.plane == "Sagittal")
g = geoms[sag_id]
by_name = sorted(g.headers, key=lambda h: h.path.name)
m = len(g.headers) // 2
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for row, (title, seq) in enumerate([("dosya adı sırası (ESKİ)", by_name), ("geometrik sıra (YENİ)", g.headers)]):
    for j, h in enumerate(seq[m - 1:m + 2]):
        axes[row, j].imshow(_read_pixels(h), cmap="gray")
        axes[row, j].set_title(f"{title}\nkonum x={h.ipp[0]:.1f} mm", fontsize=9)
        axes[row, j].axis("off")
plt.tight_layout(); plt.show()

## 2. Görüntü yönü

Her seri için satır (`r`) ve sütun (`c`) yön kosinüslerinin baskın eksenine bakıyoruz ve
görüntüyü şu standarda çeviriyoruz: satır yönü +x veya +y, sütun yönü −z (baş yukarıda)
veya +y. Böylece farklı cihazların "ters kaydettiği" seriler aynı görünür.

In [ ]:
pd.DataFrame([{
    "düzlem": g.plane, "satır ekseni": "xyz"[g.row_axis], "sütun ekseni": "xyz"[g.col_axis],
    "normal ekseni": "xyz"[g.normal_axis], "yatay çevir": g.flip_lr, "dikey çevir": g.flip_ud,
    "Laterality etiketi": g.laterality_tag, "merkez x (mm)": round(g.center_x_mm, 1),
} for g in geoms.values()])

## 3. Sağ mı sol mu?

Medial (iç) ve lateral (dış) bulgular — iki menisküs, medial/lateral OA, MCL — görüntüde
dizin tarafına göre yer değiştirir. Model bunu öğrenemez çünkü hangi diz olduğunu bilmez.
Çözüm: tüm dizleri tek tarafa (sol) getirmek.

1. Önce DICOM `Laterality` etiketi; bu veri setinde vakaların yaklaşık yarısında yok.
2. Yoksa **geometri**: görüntü merkezinin hasta x koordinatı. `x < 0` → hastanın sağında →
   **sağ diz**. Orta hatta çok yakınsa (|x| < 10 mm) karar verilmez.

Aynalama: koronal/aksiyalde sol-sağ ekseni görüntünün yatay ekseni → yatay çevirme;
sagitalde sol-sağ ekseni kesit ekseni → kesit sırası ters çevrilir.

In [ ]:
lat = resolve_laterality(list(geoms.values()))
lat

## 4. Ham görüntü → kanonik görüntü

Kanonik: sıralı, yönü standart, 160 mm FOV → 224×224, sağ diz ise aynalanmış, seri
düzeyinde %0.5–%99.5 yüzdelik pencere ile 0–255 (uint8).

In [ ]:
cfg = CacheConfig()
mirror = lat["side"] == "R"
fig, axes = plt.subplots(2, len(geoms), figsize=(3.2 * len(geoms), 6.5))
for j, (sid, g) in enumerate(geoms.items()):
    raw = _read_pixels(g.headers[len(g.headers) // 2])
    vol = load_series_volume(g, cfg.n_slices, cfg.size, cfg.fov_mm, mirror_to_left=mirror)
    axes[0, j].imshow(raw, cmap="gray"); axes[0, j].set_title(f"HAM {g.plane}\n{raw.shape[0]}x{raw.shape[1]}", fontsize=9)
    axes[1, j].imshow(vol[len(vol) // 2], cmap="gray"); axes[1, j].set_title(f"KANONİK\n{vol.shape}", fontsize=9)
    for a in axes[:, j]: a.axis("off")
plt.tight_layout(); plt.show()

## 5. Vaka → slotlar (cache'e yazılan şey)

`process_study` bir vakanın serilerini 6 slota yerleştirir (düzlem × sıvı-duyarlı mı) ve
her slottan, yığının ortadaki %80'inden eşit aralıklı 16 kesit alır. Cache'e yazılan
tam olarak budur: `uint8 [slot, 16, 224, 224]`.

In [ ]:
res = process_study(STUDY, study_rows, DATA_DIR, "test", cfg)
print(res["meta"])
vols, slots = res["volumes"], res["slots"]
fig, axes = plt.subplots(len(slots), 6, figsize=(13, 2.3 * len(slots)))
for i, (v, s) in enumerate(zip(vols, slots)):
    for j, k in enumerate(np.linspace(0, len(v) - 1, 6).astype(int)):
        axes[i, j].imshow(v[k], cmap="gray"); axes[i, j].axis("off")
        axes[i, j].set_title(f"{SLOT_NAMES[s]} · kesit {k}", fontsize=8)
plt.tight_layout(); plt.show()

## 6. 2.5D pencere: modelin gördüğü girdi

Ardışık üç kesit (artık gerçekten komşu) RGB'nin üç kanalı gibi verilir. Aşağıda bir
pencerenin kanalları ve RGB birleşimi: kenarlardaki renk saçılması komşu kesitler arası
farkı (3. boyut bilgisini) gösterir.

In [ ]:
v = vols[0].astype(np.float32) / 255
c = len(v) // 2
win = np.stack([v[c - 1], v[c], v[c + 1]], -1)
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
for j, name in enumerate(["kesit i-1 (R)", "kesit i (G)", "kesit i+1 (B)"]):
    axes[j].imshow(win[..., j], cmap="gray"); axes[j].set_title(name); axes[j].axis("off")
axes[3].imshow(win); axes[3].set_title("RGB 2.5D pencere"); axes[3].axis("off")
plt.tight_layout(); plt.show()

## Özet

* Kesit sırası geometriden, yönü yön kosinüslerinden, ölçeği mm'den alınıyor.
* Sağ dizler aynalanıyor; bu vakada etiket yoktu, taraf geometriden bulundu.
* Bozuk / eksik dosya tüm seriyi düşürmüyor; hatalar `meta["errors"]`'a yazılıyor.
* Aynı `process_study` fonksiyonu hem cache (04) hem test inference'ında (06) kullanılıyor,
  yani eğitim ve tahmin pikselleri birebir aynı yoldan geçiyor.

**Sonraki adım:** `04_cache_kaggle.ipynb` — tüm eğitim vakalarını Kaggle'da bir kez işleyip saklamak.